# Appendix: Additional Complications When Thinking Multi-Node
---

The main workshop runs on single-node multi-GPU hardware (4 GPUs in one box), so every concrete exercise stops at the boundary of one machine.  But in practice, large-scale training crosses node boundaries — and the moment it does, several things change: the communication hierarchy gains a much slower bottom layer (the network), NCCL builds different ring/tree topologies that span nodes, the profiler produces one report per rank rather than one per run, and the launch tooling shifts from `torchrun` alone to `mpirun`/`srun`-orchestrated launches.

This appendix collects the multi-node-specific material that doesn't fit in the main labs.  None of it is required to follow the workshop; it's reference for the moment you do find yourself with a multi-node job to profile.

**What's in here:**

- How to launch nsys under `mpirun`/`srun` to get one report per rank.
- How NCCL constructs rings and trees across nodes, and how to read its diagnostics.
- What multi-node profiler reports look like and how to compare across ranks.
- Common multi-node performance patterns to watch for.

## Launching nsys for Multi-Node Profiling

On compute clusters with multiple nodes and a workload manager, the profiling-command structure changes.  Two main patterns:

- **Single node, no workload manager**: `nsys profile [nsys_args] mpirun [mpirun_args] your_executable` creates one report file containing all processes.
- **Multiple nodes**: `mpirun [mpirun_args] nsys profile [nsys_args] your_executable` creates one report per MPI rank.  Use `-o report_name_%q{OMPI_COMM_WORLD_RANK}` to distinguish ranks.

To profile only specific ranks, wrap the command in a shell script:

```bash
!/bin/bash
# OMPI_COMM_WORLD_LOCAL_RANK gives the node-local rank
if [ $OMPI_COMM_WORLD_RANK -eq 0 ]; then
    nsys profile -t mpi "$@"
else
    "$@"
fi
```

**Example multi-node command with Slurm**:

```bash
srun [SRUN_ARGS] nsys profile --trace cuda,osrt,nvtx,ucx \
  --gpu-metrics-devices=cuda-visible \
  --gpu-metrics-frequency=5000 \
  --nic-metrics=true \
  --output reports/output%q{SLURM_NODEID}_%q{SLURM_PROCID} \
  --force-overwrite true ./myprogram [PROGRAM_ARGS]
```

Worth-knowing flags beyond the single-node set:

- `--nic-metrics=[true|false]`: collect NIC (network interface card) bandwidth metrics — essential for multi-node since the network is the bottleneck-of-last-resort.
- `--trace=...,ucx`: traces UCX, the transport layer typically used under MPI/NCCL for InfiniBand/RoCE communication.
- `--output reports/output%q{SLURM_NODEID}_%q{SLURM_PROCID}`: file-naming pattern that interpolates the Slurm environment so each rank gets a distinct file.

The standard ergonomic move is to symlink or copy all rank reports into one directory, then either open them as a multi-report view in the GUI (next section) or analyze them with `nsys recipe` (see Lab 3's `nsight_advanced.ipynb` for the multi-report recipe pattern, which works the same way for multi-node).

## Multi-Node Profiling Concepts (Reference)

In a multi-node cluster environment with Slurm, profiling would involve:

**Setup Overview:**
- Submit jobs using `sbatch` with a Slurm script
- Specify multiple nodes (e.g., `#SBATCH --nodes=2`)
- Each node runs multiple GPU processes
- Nsight Systems generates one report per GPU rank

**Example Multi-Node Configuration:**
- 2 nodes, 4 GPUs per node = 8 total GPUs
- Results in 8 separate `.nsys-rep` profiling reports
- Each report shows that rank's perspective of the training

**Key Differences from Single-Node:**
- **Communication:** Mix of NVLink (intra-node) and InfiniBand/Ethernet (inter-node)
- **NCCL Topology:** More complex ring and tree structures spanning nodes  
- **Synchronization:** Additional latency from cross-node communication
- **Reports:** Multiple reports that must be analyzed together

In the following sections, we'll examine what multi-node profiling reports reveal about NCCL communication patterns and distributed training dynamics.

In [ ]:
# For reference: In multi-node setup, you would collect reports with:
# !cd ../reports && tar -cvf reports.tar *.nsys-rep

After executing the above cell, you should be able to download and save the tar file by holding down Shift and right-clicking [Here](../report/reports.tar) then choosing save Link As. Once done, open the Nsight Systems and follow the below steps to use multi-report feature.

### Viewing Multiple Reports in the Same Timeline
You can open several reports in a single timeline. This could be done using one of these methods:

- **File > Open…** in the main menu, and select several report files.

- **File > New multi-report view** in the main menu, add report files that you want to open in the Multi-report view, and click the “Apply” button.


<img src="images/new-multi-report-view.png">

Multi-report view contains simple editor that allows to add/remove some report files and will load them all on a single timeline after applying that set of reports. When reports are loaded, one can use the *View Selector* to open the Multi-report view again, change the set of reports, and click on “Apply” button to reload the timeline with the new set of reports.

<img src="images/multi-report-view.png">
 
The selected set of reports can be saved as a Multi-report view document and could be opened later to load the same set again.

Now, let's have a look at the example profiler report. If you collapse the `gpuxxx` rows, you can see the number of nodes used for profiling as shown in the screenshot below.

<img src="images/p1.png">

If you look into each node, you see all the collected metrics and data, including CPU, NIC, NVTX, and CUDA traces.
<img src="images/p2.png">

Similar to single-GPU and multi-GPU profiling with Nsight Systems, you can expand the `CUDA HW` row to inspect CUDA API calls, NVTX, Kernels, NCCL communication, and memory usage, and see where the bottlenecks are.

<img src="images/p3.png">

For instance, when we hover the mouse over the specific kernel (as shown below), we can see details such as the number of grids, blocks, duration, and theoretical occupancy, among other metrics. In this case, we can see that NCCL automatically selected Tree-based AllReduce over Ring-based AllReduce. The optimization happens automatically, but you can always enforce NCCL to use Ring-based AllReduce if needed by using `export NCCL_ALGO=Ring`. 
<img src="images/p4.png">

Below is a simplified view of both the Tree and Ring structures as a reference (8 GPUs across 2 nodes).
```bash
#Tree order
              GPU0
             /    \
          GPU1    GPU4
         /   \    /   \
     GPU2  GPU3 GPU5 GPU6
                        \
                        GPU7

#Ring order
Node 0: GPU0 → GPU1 → GPU2 → GPU3
Node 1: GPU4 → GPU5 → GPU6 → GPU7
        ↑                           ↓
        ← ← ← ← ← ← ← ← ← ← ← ← ← ←
GPU0 → GPU1 → GPU2 → GPU3 → GPU4 → GPU5 → GPU6 → GPU7 → GPU0
```

By adding `export NCCL_DEBUG=INFO` to the `sbatch` script, you can enable logs that show the structure, tree levels, and information about the backend transport (P2P, SHM, IB). The logs will be written into `ddpslurm.outputxxx` file. Below are some examples illustrating the kind of output you see from the logs.

1. Ring Construction
    `Ring 00 : 0 -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 0`
    This tells you the logical ring order NCCL built for collective communication.
    - Each number is a GPU’s global rank
    - The ring loops back from last to first
    - Multiple rings may be created (e.g., Ring 00, Ring 01, …) for parallelism
2. Transport Types
``` bash
   Channel 00 : 0[0] -> 1[0] via P2P/IPC
   Channel 01 : 4[1] -> 5[1] via NET/IB
```
    This tells you how NCCL connects each GPU pair.
    - P2P = GPU-to-GPU (NVLink or PCIe)
    - SHM = shared memory (within node)
    - NET/IB = inter-node (via InfiniBand or Ethernet)
    - Channel XX = a communication channel (NCCL uses many in parallel)
3. Algorithm Selection
``` bash
   AllReduce Ring
   AllReduce Tree
```
    This tells you which algorithm NCCL used for each collective operation.

When put together, you might see the following for the example DDP code. The logs help you better understand how things work under the hood and make it easier to interpret the profiler report.

Each “Ring” line describes the communication path per ring and per channel: `NCCL INFO Ring 00 : 5 -> 0 -> 3`
This means in Ring 0, rank 5 sends to 0, which sends to 3. This is the communication path for channel 0 on this rank. Since this is printed per-rank, you can piece together full rings across ranks. For this example, there are 16 channels, each with its own ring. So we will see 16 lines as `Ring 00 – Ring 15`.

In this example code, the trees are used in NCCL’s tree-based collectives, such as reduce_scatter and broadcast.

Example log would be `Tree 0 : -1 -> 0 -> 1/4/-1`, where
``` bash
    -1 -> 0 = root is 0 (receives from no one, indicated by -1)
	0 -> 1/4 = 0 sends to ranks 1 and 4
	-1 at the end indicates no third child
```
Each tree line defines a directed communication pattern, often duplicated for upward and downward paths. You might see `Tree 0 to Tree 7`, then `Tree 4 : 4 -> 0 -> 1/-1/-1`, which is from the peer node. It indicates bidirectional trees for more balanced communication.

In the logs, you will also have data path divisions: `Channel 00/16 : 0 3 2 1 4 7 6 5`, which shows the ring order across all ranks (local + remote). In other words, in channel 0, data passes in the order of rank 0 → 3 → 2 → 1 → 4 → 7 → 6 → .

There will be information on affinity in the log too. For instance, `Setting affinity for GPU 0 to ff0000,00000000,...` sets CPU affinity for GPU 0 and `NET/0 GPU/0 GPU/3 GPU/2 GPU/1 NET/0`, indicates that NCCL sees this as a network-topology-aware ring, starting and ending at the NIC (NET/0). This is good for scaling across nodes.

You will also have information on the transport type `Channel 00/0 : 5[1] -> 0[0] [receive] via NET/Socket/0`, which tells you that on channel 0, rank 0 on node 0 receives from rank 5 on node 1, over the network. The [1] and [0] indicate the ranks and their local rank on the node.

## Advanced Analysis
Below are some of the metrics one must examine when looking at the profiler report:

<b>Key Metrics to Examine</b>
1. Communication Patterns
    - Look for regular spikes in network activity that correspond to gradient synchronization
    - These typically appear after each backward pass
2. Bandwidth Utilization
    - Check if your network is saturated (approaching theoretical bandwidth limits)
    - Identify periods of low utilization that might indicate computational bottlenecks
3. Message Sizes
    - Large messages indicate all-reduce operations for gradient synchronization
    - Small, frequent messages may indicate inefficient communication patterns
      
<b>Common Patterns</b>
1. Gradient Synchronization Barriers: Regular spikes that align with iteration boundaries
2. Wait Times: Periods where some nodes are idle while waiting for others
3. Load Imbalance: Uneven communication patterns across nodes

<b>Tips for Optimization</b>
1. Compare network activity with GPU utilization to identify if your workload is network-bound
2. Look for overlapping of computation and communication (ideal pattern)
3. Check for serialization points where all nodes must synchronize

Make sure to look for matching NVTX ranges across nodes and identify nodes that take longer to complete iterations. It is important to check whether communication occurs during computation and whether there is workload balance across the nodes. 

By following these steps, you'll get comprehensive insights into your multi-node DDP training performance, helping you identify and resolve bottlenecks for optimal scaling.



---

This is appendix material — when you're done here, return to the [main table of contents](../start_here.ipynb) or jump to whichever lab you're working on.

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.